# Credit Card Fraud Detection

This project uses machine learning to detect fraudulent transactions on an imbalanced dataset using SMOTE and classification models.

In [ ]:
# Import necessary libraries for data processing, modeling, and visualization
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from imblearn.over_sampling import SMOTE

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Upload dataset file from local system (for Google Colab)
from google.colab import files
files.upload()

In [ ]:
# Load dataset into pandas DataFrame
df = pd.read_csv("creditcard.csv")
df.head()

In [ ]:
# Analyze class distribution to check imbalance in dataset
print(df['Class'].value_counts())

sns.countplot(x='Class', data=df)
plt.title("Class Distribution")
plt.show()

In [ ]:
# Scale 'Amount' feature to normalize values for better model performance
scaler = StandardScaler()
df['Amount'] = scaler.fit_transform(df[['Amount']])

In [ ]:
# Separate features (X) and target variable (y)
X = df.drop('Class', axis=1)
y = df['Class']

# Split dataset into training and testing sets with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Apply SMOTE to handle class imbalance by oversampling minority class
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("After SMOTE:", np.bincount(y_train_res))

After SMOTE: [227451 227451]


In [ ]:
# Train Logistic Regression model on balanced dataset
lr = LogisticRegression(max_iter=2000)
lr.fit(X_train_res, y_train_res)

# Generate predictions using Logistic Regression
y_pred_lr = lr.predict(X_test)

In [ ]:
# Train Random Forest model for better performance on complex data
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_res, y_train_res)

# Generate predictions using Random Forest
y_pred_rf = rf.predict(X_test)

In [ ]:
# Function to evaluate model using classification metrics and confusion matrix
def evaluate_model(y_test, y_pred, model_name):
    print(f"\n--- {model_name} ---")
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d')
    plt.title(f"{model_name} - Confusion Matrix")
    plt.show()

In [ ]:
# Evaluate both models
evaluate_model(y_test, y_pred_lr, "Logistic Regression")
evaluate_model(y_test, y_pred_rf, "Random Forest")

In [ ]:
# Calculate ROC-AUC score to measure model performance on classification
rf_probs = rf.predict_proba(X_test)[:, 1]
roc_score = roc_auc_score(y_test, rf_probs)

print("Random Forest ROC-AUC Score:", roc_score)

Random Forest ROC-AUC Score: 0.9691768150000575
